# Load FMAPI Databricks Rates

This notebook creates the `fmapi_databricks_rates` table for Databricks-hosted foundation models:
- Llama (4 Maverick, 3.3 70B, 3.1 405B/70B/8B, 3.2 3B/1B)
- DBRX
- Mixtral 8x7B
- Embeddings (GTE, BGE Large)

**Billing types:**
- Pay-per-token (input/output)
- Provisioned throughput (entry capacity / scaling capacity)

**Source:** https://docs.databricks.com/aws/en/resources/pricing


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_FMAPI_DATABRICKS = f"{CATALOG}.{SCHEMA}.fmapi_databricks_rates"

print(f"✅ Target table: {TABLE_FMAPI_DATABRICKS}")


In [ ]:
import pandas as pd
from datetime import datetime

# =============================================================================
# FMAPI DATABRICKS RATES
# Columns: cloud, model, rate_type, dbu_rate, input_divisor, is_hourly, sku_product_type
#
# rate_type: input_token, output_token, provisioned_entry, provisioned_scaling
# input_divisor: 1000000 for tokens (per 1M), 1 for provisioned (per hour)
#
# NOTE: Pay-per-token rates are same across clouds
#       Provisioned throughput rates differ:
#       - AWS/Azure have entry capacity (lower cost starting point)
#       - GCP has NO entry capacity - only scaling capacity
# =============================================================================

FMAPI_RATES = []

def add_token_rate(model, input_rate, output_rate):
    """Add pay-per-token rates (same across all clouds)."""
    for cloud in ["AWS", "AZURE", "GCP"]:
        if input_rate:
            FMAPI_RATES.append({
                "cloud": cloud,
                "model": model,
                "rate_type": "input_token",
                "dbu_rate": input_rate,
                "input_divisor": 1000000,
                "is_hourly": False,
                "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
            })
        if output_rate:
            FMAPI_RATES.append({
                "cloud": cloud,
                "model": model,
                "rate_type": "output_token",
                "dbu_rate": output_rate,
                "input_divisor": 1000000,
                "is_hourly": False,
                "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
            })

def add_provisioned_rate(cloud, model, entry_rate, scaling_rate):
    """Add provisioned throughput rates (cloud-specific)."""
    if entry_rate:
        FMAPI_RATES.append({
            "cloud": cloud,
            "model": model,
            "rate_type": "provisioned_entry",
            "dbu_rate": entry_rate,
            "input_divisor": 1,
            "is_hourly": True,
            "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        })
    if scaling_rate:
        FMAPI_RATES.append({
            "cloud": cloud,
            "model": model,
            "rate_type": "provisioned_scaling",
            "dbu_rate": scaling_rate,
            "input_divisor": 1,
            "is_hourly": True,
            "sku_product_type": "SERVERLESS_REAL_TIME_INFERENCE",
        })

# =============================================================================
# PAY-PER-TOKEN RATES (same across all clouds)
# =============================================================================

add_token_rate("llama-4-maverick", input_rate=7.143, output_rate=21.429)
add_token_rate("llama-3-3-70b", input_rate=7.143, output_rate=21.429)
add_token_rate("gpt-oss-120b", input_rate=2.143, output_rate=8.571)
add_token_rate("gemma-3-12b", input_rate=2.143, output_rate=7.143)
add_token_rate("llama-3-1-8b", input_rate=2.143, output_rate=6.429)
add_token_rate("gpt-oss-20b", input_rate=1.0, output_rate=4.286)
# Llama 3.2 3B and 1B are provisioned only - no pay-per-token
add_token_rate("gte", input_rate=1.857, output_rate=None)
add_token_rate("bge-large", input_rate=1.429, output_rate=None)

print(f"✅ Added pay-per-token rates")

# =============================================================================
# PROVISIONED THROUGHPUT RATES - AWS (has entry capacity)
# =============================================================================

add_provisioned_rate("AWS", "llama-4-maverick", entry_rate=85.714, scaling_rate=85.714)
add_provisioned_rate("AWS", "llama-3-3-70b", entry_rate=85.714, scaling_rate=342.857)
add_provisioned_rate("AWS", "gpt-oss-120b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("AWS", "gemma-3-12b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("AWS", "llama-3-1-8b", entry_rate=53.571, scaling_rate=106.0)
add_provisioned_rate("AWS", "gpt-oss-20b", entry_rate=53.571, scaling_rate=53.571)
add_provisioned_rate("AWS", "llama-3-2-3b", entry_rate=46.429, scaling_rate=92.857)
add_provisioned_rate("AWS", "llama-3-2-1b", entry_rate=42.857, scaling_rate=85.714)
add_provisioned_rate("AWS", "gte", entry_rate=20.0, scaling_rate=20.0)
add_provisioned_rate("AWS", "bge-large", entry_rate=24.0, scaling_rate=24.0)

print(f"✅ Added AWS provisioned rates")

# =============================================================================
# PROVISIONED THROUGHPUT RATES - AZURE (has entry capacity, same as AWS)
# =============================================================================

add_provisioned_rate("AZURE", "llama-4-maverick", entry_rate=85.714, scaling_rate=85.714)
add_provisioned_rate("AZURE", "llama-3-3-70b", entry_rate=85.714, scaling_rate=342.857)
add_provisioned_rate("AZURE", "gpt-oss-120b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("AZURE", "gemma-3-12b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("AZURE", "llama-3-1-8b", entry_rate=53.571, scaling_rate=106.0)
add_provisioned_rate("AZURE", "gpt-oss-20b", entry_rate=53.571, scaling_rate=53.571)
add_provisioned_rate("AZURE", "llama-3-2-3b", entry_rate=46.429, scaling_rate=92.857)
add_provisioned_rate("AZURE", "llama-3-2-1b", entry_rate=42.857, scaling_rate=85.714)
add_provisioned_rate("AZURE", "gte", entry_rate=20.0, scaling_rate=20.0)
add_provisioned_rate("AZURE", "bge-large", entry_rate=24.0, scaling_rate=24.0)

print(f"✅ Added Azure provisioned rates")

# =============================================================================
# PROVISIONED THROUGHPUT RATES - GCP (NO entry capacity - entry = scaling)
# Entry capacity not available on GCP
# =============================================================================

add_provisioned_rate("GCP", "llama-4-maverick", entry_rate=85.714, scaling_rate=85.714)
add_provisioned_rate("GCP", "llama-3-3-70b", entry_rate=342.857, scaling_rate=342.857)  # No entry discount
add_provisioned_rate("GCP", "gpt-oss-120b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("GCP", "gemma-3-12b", entry_rate=71.429, scaling_rate=71.429)
add_provisioned_rate("GCP", "llama-3-1-8b", entry_rate=106.0, scaling_rate=106.0)  # No entry discount
add_provisioned_rate("GCP", "gpt-oss-20b", entry_rate=53.571, scaling_rate=53.571)
add_provisioned_rate("GCP", "llama-3-2-3b", entry_rate=92.857, scaling_rate=92.857)  # No entry discount
add_provisioned_rate("GCP", "llama-3-2-1b", entry_rate=85.714, scaling_rate=85.714)  # No entry discount
add_provisioned_rate("GCP", "gte", entry_rate=20.0, scaling_rate=20.0)
add_provisioned_rate("GCP", "bge-large", entry_rate=24.0, scaling_rate=24.0)

print(f"✅ Added GCP provisioned rates (no entry capacity discount)")

print(f"\n✅ Total: {len(FMAPI_RATES)} FMAPI Databricks rates")


In [ ]:
# =============================================================================
# CREATE DATAFRAME AND SAVE
# =============================================================================

# Add timestamp
for row in FMAPI_RATES:
    row["updated_at"] = datetime.utcnow().isoformat()

df = pd.DataFrame(FMAPI_RATES)

print(f"📊 Models: {df['model'].unique().tolist()}")
print(f"📊 Rate types: {df['rate_type'].unique().tolist()}")
print(f"📊 Total rates: {len(df)}")
display(df)

# Save to table
spark_df = spark.createDataFrame(df)
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_FMAPI_DATABRICKS)

print(f"\n✅ Saved to {TABLE_FMAPI_DATABRICKS}")
